In [240]:
import json
import os
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import (
    average_precision_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


In [241]:
TRAIN_FEAT_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/train_features.csv"
TRAIN_TARGET_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/customer_clv_train.csv"
BEST_PARAMS_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/best_params.json"

In [242]:
# Load train data and build modeling table
required_paths = [TRAIN_FEAT_PATH, TRAIN_TARGET_PATH]
missing_required = [p for p in required_paths if not os.path.exists(p)]
if missing_required:
    raise FileNotFoundError(f"Missing required files: {missing_required}")

train_features = pd.read_csv(TRAIN_FEAT_PATH)
train_target = pd.read_csv(TRAIN_TARGET_PATH)

train_df = train_target.merge(train_features, on="cust_id", how="inner", validate="one_to_one")
feature_cols = [c for c in train_df.columns if c not in ["cust_id", "revenue_2018_2019"]]
X = train_df[feature_cols].copy()
y = train_df["revenue_2018_2019"].to_numpy()
y_bin = (y > 0).astype(int)

print("train_df shape:", train_df.shape)
print("X shape:", X.shape)
print("positive ratio:", y_bin.mean())


train_df shape: (116591, 24)
X shape: (116591, 22)
positive ratio: 0.36587729756156134


In [243]:
# Load tuned params if available; fallback to defaults
clf_params = {
    "iterations": 400,
    "depth": 6,
    "learning_rate": 0.05,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "l2_leaf_reg": 7,
    "random_state": 42,
    "verbose": False,
}

reg_params = {
    "iterations": 400,
    "depth": 6,
    "l2_leaf_reg": 7,
    "learning_rate": 0.05,
    "loss_function": "MAE",
    "random_state": 42,
    "verbose": False,
}

#if os.path.exists(BEST_PARAMS_PATH):
    #with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
        #best_params = json.load(f)
    #clf_params.update(best_params.get("classifier", {}).get("best_params", {}))
    #reg_params.update(best_params.get("regressor", {}).get("best_params", {}))
    #print(f"Loaded tuned params from: {BEST_PARAMS_PATH}")
#else:
    #print("best_params.json not found. Using defaults.")

print("Classifier params:", clf_params)
print("Regressor params:", reg_params)


Classifier params: {'iterations': 400, 'depth': 6, 'learning_rate': 0.05, 'loss_function': 'Logloss', 'eval_metric': 'AUC', 'l2_leaf_reg': 7, 'random_state': 42, 'verbose': False}
Regressor params: {'iterations': 400, 'depth': 6, 'l2_leaf_reg': 7, 'learning_rate': 0.05, 'loss_function': 'MAE', 'random_state': 42, 'verbose': False}


In [246]:
# Holdout validation
X_tr, X_val, y_tr, y_val, yb_tr, yb_val = train_test_split(
    X,
    y,
    y_bin,
    test_size=0.2,
    random_state=42,
    stratify=y_bin,
)

# Stage 1: classifier
clf = CatBoostClassifier(**clf_params)
clf.fit(X_tr, yb_tr)
tr_prob = clf.predict_proba(X_tr)[:, 1]
val_prob = clf.predict_proba(X_val)[:, 1]

# Stage 2: regressor with classifier output as feature
X_tr_reg = X_tr.copy()
X_val_reg = X_val.copy()
X_tr_reg["prob_return"] = tr_prob
X_val_reg["prob_return"] = val_prob

reg = CatBoostRegressor(**reg_params)


#reg.fit(X_tr_reg, np.log1p(y_tr))
weights = 1 + 3 * tr_prob   # prueba también 1 + 5 * tr_prob

reg.fit(X_tr_reg, y_tr, sample_weight=weights)

#val_log = reg.predict(X_val_reg)
val = reg.predict(X_val_reg)

#y_pred = np.clip(np.expm1(val_log), 0, None)
#y_pred = np.clip(val, 0, np.percentile(y_tr, 98))
y_pred = np.clip(val, 0, None)

# Baseline for comparison (predict train mean)
baseline_pred = np.full_like(y_val, fill_value=y_tr.mean(), dtype=float)

# Classification metrics
auc = roc_auc_score(yb_val, val_prob)
ap = average_precision_score(yb_val, val_prob)

# Regression metrics (on all customers)
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
r2 = r2_score(y_val, y_pred)

# Regression metrics on positive spenders only
pos_mask = y_val > 0
if pos_mask.sum() > 0:
    mae_pos = mean_absolute_error(y_val[pos_mask], y_pred[pos_mask])
    rmse_pos = np.sqrt(mean_squared_error(y_val[pos_mask], y_pred[pos_mask]))
else:
    mae_pos = np.nan
    rmse_pos = np.nan

baseline_mae = mean_absolute_error(y_val, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_pred))


In [247]:
reg_pred = np.clip(reg.predict(X_val_reg), 0, None)

alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.8, 0.9]

for alpha in alphas:
    y_pred = reg_pred * (alpha * val_prob + (1 - alpha))
    
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    
    print(f"alpha={alpha} | MAE={mae:.4f} | RMSE={rmse:.4f}")

alpha=0.1 | MAE=61.7675 | RMSE=132.8677
alpha=0.2 | MAE=61.7660 | RMSE=133.1639
alpha=0.3 | MAE=61.7733 | RMSE=133.4722
alpha=0.4 | MAE=61.7876 | RMSE=133.7926
alpha=0.5 | MAE=61.8110 | RMSE=134.1248
alpha=0.7 | MAE=61.8872 | RMSE=134.8249
alpha=0.8 | MAE=61.9376 | RMSE=135.1925
alpha=0.9 | MAE=61.9953 | RMSE=135.5717


In [235]:
from sklearn.feature_selection import VarianceThreshold

# 1) filtro por baja varianza (casi constantes)
#vt = VarianceThreshold(threshold=1e-5)
#X_tr_v = pd.DataFrame(vt.fit_transform(X_tr), columns=X_tr.columns[vt.get_support()])
#X_val_v = pd.DataFrame(vt.transform(X_val), columns=X_tr_v.columns)
X_tr_v = pd.DataFrame(X_tr, columns=X_tr.columns)
X_val_v = pd.DataFrame(X_val, columns=X_tr_v.columns)


# 2) reentrenar con columnas filtradas
clf.fit(X_tr_v, yb_tr)
val_prob = clf.predict_proba(X_val_v)[:, 1]

X_tr_reg_v = X_tr_v.copy()
X_val_reg_v = X_val_v.copy()
X_tr_reg_v["prob_return"] = clf.predict_proba(X_tr_v)[:, 1]
X_val_reg_v["prob_return"] = val_prob

reg.fit(X_tr_reg_v, y_tr)

# 3) importancia del regresor
imp = pd.DataFrame({
    "feature": X_tr_reg_v.columns,
    "importance": reg.get_feature_importance(type="PredictionValuesChange")
}).sort_values('importance')
print(imp)  # las menos útiles


                      feature  importance
14             brand_Converse    0.129111
12            prod_type_1_men    0.233179
5          web_only_purchases    0.260717
18              brand_Tamaris    0.338655
4               delivery_time    0.408546
13          prod_type_1_women    0.457957
20                return_rate    0.479304
19             returned_items    0.494835
11          prod_type_1_girls    0.577962
7      customer_lifespan_days    0.593593
16                 brand_Geox    0.594996
10           prod_type_1_boys    0.687073
8                 tenure_days    0.751329
2                mean_revenue    0.883602
21         active_month_ratio    0.917982
17     brand_STONES and BONES    1.028603
15                brand_Gabor    1.100845
6                recency_days    1.296745
3                  n_products    1.970446
22          revenue_per_order    2.480522
1   customer_purchases_number    3.386118
9               active_months    3.958083
0            customer_revenue   14

In [248]:
metrics = pd.DataFrame(
    [
        ["classifier_auc", auc],
        ["classifier_avg_precision", ap],
        ["reg_mae_all", mae],
        ["reg_rmse_all", rmse],
        ["reg_r2_all", r2],
        ["reg_mae_positive_only", mae_pos],
        ["reg_rmse_positive_only", rmse_pos],
        ["baseline_mae_all", baseline_mae],
        ["baseline_rmse_all", baseline_rmse],
    ],
    columns=["metric", "value"],
)

print("Validation metrics (holdout):")
print(metrics.to_string(index=False))

improvement_mae = baseline_mae - mae
improvement_rmse = baseline_rmse - rmse
print(f"\nImprovement vs baseline (MAE): {improvement_mae:.6f}")
print(f"Improvement vs baseline (RMSE): {improvement_rmse:.6f}")


Validation metrics (holdout):
                  metric      value
          classifier_auc   0.724840
classifier_avg_precision   0.639418
             reg_mae_all  61.995257
            reg_rmse_all 135.571671
              reg_r2_all   0.145809
   reg_mae_positive_only 152.087677
  reg_rmse_positive_only 214.872729
        baseline_mae_all  92.969421
       baseline_rmse_all 143.454794

Improvement vs baseline (MAE): 30.974164
Improvement vs baseline (RMSE): 7.883124


In [ ]:
# Optional: save metrics
METRICS_OUT_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/validation_metrics.csv"
metrics.to_csv(METRICS_OUT_PATH, index=False)
print(f"Saved metrics to: {METRICS_OUT_PATH}")
